In [ ]:
import pandas as pd


TARGET_SINGLE = 1000_000
RANDOM_STATE = 42


df_merged = pd.read_csv(
    r"D:\feature 4\experence3\merged_text_tags_grouped.csv",
    encoding="utf-8-sig"
)


df_merged["text"] = df_merged["text"].fillna("").astype(str).str.strip()
df_merged["Tags"] = df_merged["Tags"].fillna("").astype(str).str.strip()

df_merged = df_merged[
    (df_merged["text"] != "") &
    (df_merged["Tags"] != "")
].copy()


df_merged["num_tags"] = df_merged["Tags"].apply(lambda x: len(str(x).split()))


single_tag_df = df_merged[df_merged["num_tags"] == 1].copy()
multi_tag_df  = df_merged[df_merged["num_tags"] >= 2].copy()


single_tag_df = single_tag_df.drop_duplicates(subset=["text", "Tags"]).copy()
multi_tag_df  = multi_tag_df.drop_duplicates(subset=["text", "Tags"]).copy()

# =========================
# فلترة جودة النص
# =========================
for df in [single_tag_df, multi_tag_df]:
    df["text_len"] = df["text"].apply(len)
    df["word_count"] = df["text"].apply(lambda x: len(x.split()))

single_tag_df = single_tag_df[
    (single_tag_df["word_count"] >= 12) &
    (single_tag_df["text_len"] >= 50) &
    (single_tag_df["text_len"] <= 5000)
].copy()

multi_tag_df = multi_tag_df[
    (multi_tag_df["word_count"] >= 8) &
    (multi_tag_df["text_len"] >= 40) &
    (multi_tag_df["text_len"] <= 5000)
].copy()

print("Single-tag rows after cleaning:", len(single_tag_df))
print("Multi-tag rows after cleaning:", len(multi_tag_df))

# =========================
# سحب عشوائي من single-tag
# =========================
target_single_safe = min(TARGET_SINGLE, len(single_tag_df))
single_selected = single_tag_df.sample(
    n=target_single_safe,
    random_state=RANDOM_STATE
).copy()

print("Selected single-tag rows:", len(single_selected))

# =========================
# دمج single المختار + كل multi-tag
# =========================
final_df = pd.concat([single_selected, multi_tag_df], ignore_index=True)

# =========================
# خلط البيانات
# =========================
final_df = final_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# =========================
# الاحتفاظ فقط بالأعمدة المهمة
# =========================
final_df = final_df[["text", "Tags", "num_tags"]].copy()

# =========================
# حفظ الملف النهائي
# =========================
final_df.to_csv(
    r"D:\feature 4\experence3\merged_text_tags_final_balanced.csv",
    index=False,
    encoding="utf-8-sig"
)



Single-tag rows after cleaning: 1643861
Multi-tag rows after cleaning: 233817
Selected single-tag rows: 1000000


In [2]:
# =========================
# طباعة النتائج
# =========================
print("\nFINAL RESULTS")
print("Final total rows:", len(final_df))

print("\nSingle-tag count in final:")
print((final_df["num_tags"] == 1).sum())

print("\n2+ tags count in final:")
print((final_df["num_tags"] >= 2).sum())

print("\nDistribution by num_tags:")
print(final_df["num_tags"].value_counts().sort_index())

print("\nSample rows:")
print(final_df.head())


FINAL RESULTS
Final total rows: 933817

Single-tag count in final:
700000

2+ tags count in final:
233817

Distribution by num_tags:
num_tags
1    700000
2    213943
3     18578
4      1245
5        51
Name: count, dtype: int64

Sample rows:
                                                text                  Tags  \
0  servicestack rest api and cors anyone know if ...                api c#   
1  how do you know when two objects can communica...                python   
2  gson is there an easier way to serialize a map...                  java   
3  regex to extract words in quotes out of a stri...  design-patterns java   
4  how can i close my software in a safe way up t...              api java   

   num_tags  
0         2  
1         1  
2         1  
3         2  
4         2  
